# TubeWatch Transcript Storage Offline Tester

本 Notebook 只使用临时目录、正式 Python API 和可控 fake TubeScribe 结果，离线验证 schema、migration、transcript repository 与 processing 事务一致性。

每个测试会输出：测试名称、验证目的、实际结果、期望结果和 PASS/FAIL。测试失败时会在对应项目立即停止，并保留具体的实际值与期望值。

In [1]:
from datetime import UTC, datetime, timedelta
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory
import sqlite3

from tubewatch import (
    AmbiguousTranscriptError, TubeScribeResult, VideoItem,
    cleanup_test_videos, get_transcript, initialize_state_database,
    list_transcripts_for_video, process_pending_videos,
    record_discovered_videos, save_transcript,
)
from tubewatch.exceptions import (
    StateStorageError, TubeScribeMembersOnlyError,
    TubeScribeNoSubtitlesError, TubeScribeProcessingError,
)

def video(video_id):
    return VideoItem(video_id, f'Title {video_id}', f'https://www.youtube.com/watch?v={video_id}', None, None, None)

def discover(db, source, *video_ids):
    return record_discovered_videos(
        db, source_type='channel', source_url=source,
        videos=[video(item) for item in video_ids], checked_at=datetime.now(UTC),
    )

def start_section(title, purpose):
    print(f'\n=== {title} ===')
    print(f'范围: {purpose}')

def report_test(name, purpose, *, actual, expected, passed):
    status = 'PASS' if passed else 'FAIL'
    print(f'\n[{status}] {name}')
    print(f'  验证: {purpose}')
    print(f'  实际: {actual!r}')
    print(f'  期望: {expected!r}')
    if not passed:
        raise AssertionError(f'{name} failed: actual={actual!r}, expected={expected!r}')

def report_equal(name, purpose, actual, expected):
    report_test(name, purpose, actual=actual, expected=expected, passed=actual == expected)

def finish_section(title, test_count):
    print(f'\n✅ {title}: {test_count}/{test_count} tests passed.')


In [2]:
# New schema, foreign keys, legacy upgrade, idempotence, and rollback.
start_section(
    'Schema and migrations',
    '验证新库结构、外键保护、旧库升级、重复初始化和 migration 失败时的事务回滚。',
)
legacy_schema = '''
CREATE TABLE sources (source_type TEXT NOT NULL, source_url TEXT NOT NULL, last_checked_at TEXT NOT NULL, PRIMARY KEY(source_type, source_url));
CREATE TABLE discovered_videos (source_type TEXT NOT NULL, source_url TEXT NOT NULL, video_id TEXT NOT NULL, title TEXT NOT NULL, url TEXT NOT NULL, published_at TEXT, channel_id TEXT, channel_name TEXT, first_seen_at TEXT NOT NULL, last_seen_at TEXT NOT NULL, PRIMARY KEY(source_type, source_url, video_id), FOREIGN KEY(source_type, source_url) REFERENCES sources(source_type, source_url) ON DELETE CASCADE);
CREATE TABLE processing_records (source_type TEXT NOT NULL, source_url TEXT NOT NULL, video_id TEXT NOT NULL, status TEXT NOT NULL CHECK(status IN ('pending','succeeded','failed','no_subtitles','members_only')), attempt_count INTEGER NOT NULL DEFAULT 0, last_attempt_at TEXT, raw_path TEXT, cleaned_path TEXT, language_code TEXT, is_automatic INTEGER, error_message TEXT, PRIMARY KEY(source_type, source_url, video_id), FOREIGN KEY(source_type, source_url,video_id) REFERENCES discovered_videos(source_type,source_url,video_id) ON DELETE CASCADE);
INSERT INTO sources VALUES ('channel','legacy','2025-01-01T00:00:00+00:00');
INSERT INTO discovered_videos VALUES ('channel','legacy','legacy-video','Legacy','https://example.invalid',NULL,NULL,NULL,'2025-01-01T00:00:00+00:00','2025-01-01T00:00:00+00:00');
INSERT INTO processing_records VALUES ('channel','legacy','legacy-video','succeeded',1,'2025-01-01T00:00:00+00:00','raw','cleaned','zh',0,NULL);
'''
with TemporaryDirectory() as temporary:
    root = Path(temporary)
    fresh = root / 'fresh.sqlite3'
    initialize_state_database(fresh)
    connection = sqlite3.connect(fresh)
    connection.execute('PRAGMA foreign_keys = ON')
    try:
        tables = {row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        required_tables = {'videos', 'transcripts', 'schema_migrations'}
        report_test(
            'Fresh database creates transcript tables',
            '初始化全新数据库时会创建全局视频、字幕和 migration 版本表。',
            actual=sorted(required_tables & tables), expected=sorted(required_tables),
            passed=required_tables <= tables,
        )
        foreign_keys = connection.execute('PRAGMA foreign_keys').fetchone()[0]
        report_equal(
            'SQLite foreign keys are enabled',
            '测试连接确实启用外键，因此后续约束测试有效。',
            foreign_keys, 1,
        )
        try:
            connection.execute("INSERT INTO transcripts (video_id,language_code,source_kind,cleaned_text,cleaned_content_hash,character_count,created_at,updated_at) VALUES ('missing','und','unknown','x','h',1,'t','t')")
        except sqlite3.IntegrityError as error:
            connection.rollback()
            report_equal(
                'Transcript requires an existing video',
                '不能为 videos 表中不存在的 video_id 插入字幕。',
                type(error).__name__, 'IntegrityError',
            )
        else:
            raise AssertionError('transcript foreign key did not fire')
    finally:
        connection.close()

    legacy = root / 'legacy.sqlite3'
    connection = sqlite3.connect(legacy); connection.executescript(legacy_schema); connection.close()
    initialize_state_database(legacy); initialize_state_database(legacy)
    connection = sqlite3.connect(legacy)
    try:
        migrated_video = connection.execute('SELECT title FROM videos').fetchone()
        report_equal(
            'Legacy video metadata is preserved',
            '旧 discovered_videos 数据升级后会回填到全局 videos 表。',
            migrated_video, ('Legacy',),
        )
        migrated_job = connection.execute('SELECT status, transcript_id FROM processing_records').fetchone()
        report_equal(
            'Unverifiable legacy success returns to pending',
            '旧 succeeded 记录没有权威 transcript，升级后必须等待重新处理且不链接 transcript。',
            migrated_job, ('pending', None),
        )
        migration_count = connection.execute('SELECT COUNT(*) FROM schema_migrations').fetchone()[0]
        report_equal(
            'Repeated initialization is idempotent',
            '连续初始化两次不会重复执行或重复登记 migration。',
            migration_count, 2,
        )
    finally:
        connection.close()

    broken = root / 'broken.sqlite3'
    connection = sqlite3.connect(broken); connection.executescript(legacy_schema + 'CREATE TABLE videos (bad TEXT);'); connection.close()
    try:
        initialize_state_database(broken)
    except StateStorageError as error:
        report_equal(
            'Broken migration reports a storage error',
            '目标 schema 冲突时初始化应明确失败，而不是静默重建数据库。',
            type(error).__name__, 'StateStorageError',
        )
    else:
        raise AssertionError('broken migration unexpectedly succeeded')
    connection = sqlite3.connect(broken)
    try:
        original_status = connection.execute('SELECT status FROM processing_records').fetchone()
        migration_table = connection.execute("SELECT name FROM sqlite_master WHERE name='schema_migrations'").fetchone()
        report_equal(
            'Failed migration rolls back legacy data changes',
            'migration 中途失败后，原 processing 状态仍保持升级前的 succeeded。',
            original_status, ('succeeded',),
        )
        report_equal(
            'Failed migration rolls back schema changes',
            'migration 中途失败后，不会留下 schema_migrations 半成品表。',
            migration_table, None,
        )
    finally:
        connection.close()
finish_section('Schema and migrations', 9)



=== Schema and migrations ===
范围: 验证新库结构、外键保护、旧库升级、重复初始化和 migration 失败时的事务回滚。

[PASS] Fresh database creates transcript tables
  验证: 初始化全新数据库时会创建全局视频、字幕和 migration 版本表。
  实际: ['schema_migrations', 'transcripts', 'videos']
  期望: ['schema_migrations', 'transcripts', 'videos']

[PASS] SQLite foreign keys are enabled
  验证: 测试连接确实启用外键，因此后续约束测试有效。
  实际: 1
  期望: 1

[PASS] Transcript requires an existing video
  验证: 不能为 videos 表中不存在的 video_id 插入字幕。
  实际: 'IntegrityError'
  期望: 'IntegrityError'

[PASS] Legacy video metadata is preserved
  验证: 旧 discovered_videos 数据升级后会回填到全局 videos 表。
  实际: ('Legacy',)
  期望: ('Legacy',)

[PASS] Unverifiable legacy success returns to pending
  验证: 旧 succeeded 记录没有权威 transcript，升级后必须等待重新处理且不链接 transcript。
  实际: ('pending', None)
  期望: ('pending', None)

[PASS] Repeated initialization is idempotent
  验证: 连续初始化两次不会重复执行或重复登记 migration。
  实际: 2
  期望: 2

[PASS] Broken migration reports a storage error
  验证: 目标 schema 冲突时初始化应明确失败，而不是静默重建数据库。
  实际: 'StateStorageError'
 

In [3]:
# Transcript insert/read/upsert/hash/count/timestamps/multilingual/ambiguity/cascade.
start_section(
    'Transcript repository',
    '验证字幕保存与读取、内容元数据、幂等 upsert、多语言选择和级联删除。',
)
with TemporaryDirectory() as temporary:
    db = Path(temporary) / 'repository.sqlite3'
    source = 'https://www.youtube.com/@repository/videos'
    discover(db, source, 'repo-video')
    first_time = datetime(2025, 1, 1, tzinfo=UTC)
    first = save_transcript(db, video_id='repo-video', language_code='zh', source_kind='manual', cleaned_text='第一版', raw_file_path='raw/repo.zh.vtt', saved_at=first_time)
    report_equal(
        'Saved transcript can be read back',
        '首次保存返回的 TranscriptRecord 与随后从 SQLite 读取的记录完全一致。',
        get_transcript(db, 'repo-video'), first,
    )
    expected_hash = sha256('第一版'.encode('utf-8')).hexdigest()
    report_equal(
        'Cleaned text SHA-256 is calculated correctly',
        'repository 根据 UTF-8 正文计算并保存确定的 SHA-256。',
        first.cleaned_content_hash, expected_hash,
    )
    report_equal(
        'Character count matches cleaned text',
        'character_count 保存 Python 字符数，而不是 UTF-8 字节数。',
        first.character_count, len('第一版'),
    )
    unchanged = save_transcript(db, video_id='repo-video', language_code='zh', source_kind='manual', cleaned_text='第一版', raw_file_path='raw/repo.zh.vtt', saved_at=first_time + timedelta(hours=1))
    unchanged_identity = (unchanged.id, unchanged.updated_at)
    report_equal(
        'Identical upsert is a no-op',
        '相同语言、来源、正文和路径再次保存时复用 ID，且 updated_at 不变化。',
        unchanged_identity, (first.id, first.updated_at),
    )
    updated = save_transcript(db, video_id='repo-video', language_code='zh', source_kind='manual', cleaned_text='第二版正文', raw_file_path='raw/repo.zh.vtt', saved_at=first_time + timedelta(hours=2))
    report_test(
        'Changed content updates the existing transcript',
        '同一唯一键的正文变化时复用 transcript ID，并推进 updated_at。',
        actual={'same_id': updated.id == first.id, 'timestamp_advanced': updated.updated_at > first.updated_at},
        expected={'same_id': True, 'timestamp_advanced': True},
        passed=updated.id == first.id and updated.updated_at > first.updated_at,
    )
    save_transcript(db, video_id='repo-video', language_code='en', source_kind='auto_generated', cleaned_text='English')
    transcript_keys = [
        (item.language_code, item.source_kind)
        for item in list_transcripts_for_video(db, 'repo-video')
    ]
    report_equal(
        'One video can store multiple transcript variants',
        '中文人工字幕和英文自动字幕可以分别保存，不互相覆盖。',
        sorted(transcript_keys), [('en', 'auto_generated'), ('zh', 'manual')],
    )
    try:
        get_transcript(db, 'repo-video')
    except AmbiguousTranscriptError as error:
        report_equal(
            'Ambiguous read requires explicit selection',
            '一个视频有多条字幕时，不带语言/来源条件的读取必须报歧义错误。',
            type(error).__name__, 'AmbiguousTranscriptError',
        )
    else:
        raise AssertionError('ambiguous transcript selection was silent')
    missing_transcript = get_transcript(db, 'repo-video', 'missing')
    report_equal(
        'Missing language returns None',
        '明确查询不存在的语言时返回 None，而不是随机返回其他字幕。',
        missing_transcript, None,
    )
    connection = sqlite3.connect(db)
    connection.execute('PRAGMA foreign_keys = ON')
    try:
        connection.execute("DELETE FROM discovered_videos WHERE video_id='repo-video'")
        connection.execute("DELETE FROM videos WHERE video_id='repo-video'")
        connection.commit()
        remaining_counts = {
            'videos': connection.execute('SELECT COUNT(*) FROM videos').fetchone()[0],
            'transcripts': connection.execute('SELECT COUNT(*) FROM transcripts').fetchone()[0],
        }
        report_equal(
            'Deleting a video cascades to its transcripts',
            '移除最后一条来源发现后删除全局 video，会由外键同时删除关联 transcript。',
            remaining_counts, {'videos': 0, 'transcripts': 0},
        )
    finally:
        connection.close()
finish_section('Transcript repository', 9)



=== Transcript repository ===
范围: 验证字幕保存与读取、内容元数据、幂等 upsert、多语言选择和级联删除。

[PASS] Saved transcript can be read back
  验证: 首次保存返回的 TranscriptRecord 与随后从 SQLite 读取的记录完全一致。
  实际: TranscriptRecord(id=1, video_id='repo-video', language_code='zh', source_kind='manual', format='plain_text', cleaned_text='第一版', cleaner_name='TubeScribe', cleaner_version=None, source_content_hash=None, cleaned_content_hash='812f7047193410c913d8e95673ba09353bd79236e4642a0a08f97df36b4c5788', character_count=3, word_count=None, raw_file_path='raw/repo.zh.vtt', created_at=datetime.datetime(2025, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2025, 1, 1, 0, 0, tzinfo=datetime.timezone.utc))
  期望: TranscriptRecord(id=1, video_id='repo-video', language_code='zh', source_kind='manual', format='plain_text', cleaned_text='第一版', cleaner_name='TubeScribe', cleaner_version=None, source_content_hash=None, cleaned_content_hash='812f7047193410c913d8e95673ba09353bd79236e4642a0a08f97df36b4c5788', characte


[PASS] One video can store multiple transcript variants
  验证: 中文人工字幕和英文自动字幕可以分别保存，不互相覆盖。
  实际: [('en', 'auto_generated'), ('zh', 'manual')]
  期望: [('en', 'auto_generated'), ('zh', 'manual')]

[PASS] Ambiguous read requires explicit selection
  验证: 一个视频有多条字幕时，不带语言/来源条件的读取必须报歧义错误。
  实际: 'AmbiguousTranscriptError'
  期望: 'AmbiguousTranscriptError'

[PASS] Missing language returns None
  验证: 明确查询不存在的语言时返回 None，而不是随机返回其他字幕。
  实际: None
  期望: None

[PASS] Deleting a video cascades to its transcripts
  验证: 移除最后一条来源发现后删除全局 video，会由外键同时删除关联 transcript。
  实际: {'videos': 0, 'transcripts': 0}
  期望: {'videos': 0, 'transcripts': 0}

✅ Transcript repository: 9/9 tests passed.


In [4]:
# Offline fake TubeScribe success, idempotence, terminal outcomes, and DB rollback.
start_section(
    'Processing integration',
    '用 fake TubeScribe 验证成功落库、跨来源幂等、业务终态和数据库写入失败回滚。',
)
with TemporaryDirectory() as temporary:
    root = Path(temporary); db = root / 'processing.sqlite3'
    raw = root / 'output' / 'raw'; cleaned = root / 'output' / 'cleaned'
    raw.mkdir(parents=True); cleaned.mkdir(parents=True)
    source_one = 'https://www.youtube.com/@processing-one/videos'
    source_two = 'https://www.youtube.com/@processing-two/videos'
    discover(db, source_one, 'ok')

    def success_processor(url, *, raw_directory, cleaned_directory):
        video_id = url.split('=')[-1]
        raw_path = Path(raw_directory) / f'{video_id}.zh.vtt'
        cleaned_path = Path(cleaned_directory) / f'{video_id}.zh.txt'
        raw_path.write_text('WEBVTT\n', encoding='utf-8')
        cleaned_path.write_text('清理后的正文', encoding='utf-8')
        return TubeScribeResult(video_id, raw_path, cleaned_path, 'zh', False)

    first_batch = process_pending_videos(db, raw, cleaned, source_url=source_one, video_ids=['ok'], processor=success_processor)
    item = first_batch.results[0]
    success_state = {
        'status': item.status,
        'transcript_saved': item.transcript_saved,
        'has_transcript_id': bool(item.transcript_id),
    }
    report_equal(
        'Successful processing links a saved transcript',
        'fake processor 成功后，job 标为 succeeded，并明确链接已保存的 transcript。',
        success_state,
        {'status': 'succeeded', 'transcript_saved': True, 'has_transcript_id': True},
    )
    report_equal(
        'Raw VTT path is stored relative to output root',
        '数据库/结果只保存可迁移的 raw 相对路径，不保存临时目录绝对路径。',
        item.raw_file_path, 'raw/ok.zh.vtt',
    )
    transcript = get_transcript(db, 'ok')
    stored_text = transcript.cleaned_text if transcript else None
    report_equal(
        'Cleaned TXT content becomes authoritative database text',
        'fake processor 写出的 cleaned TXT 会被读取并保存到 SQLite transcript。',
        stored_text, '清理后的正文',
    )
    stored_count = transcript.character_count if transcript else None
    report_equal(
        'Processed transcript stores the correct character count',
        'processing workflow 保存的 character_count 与数据库正文字符数一致。',
        stored_count, len('清理后的正文'),
    )

    discover(db, source_two, 'ok')
    second_batch = process_pending_videos(db, raw, cleaned, source_url=source_two, video_ids=['ok'], processor=success_processor)
    report_equal(
        'The same video in another source reuses its transcript',
        '同一 video_id 从第二个来源处理时链接相同的全局 transcript ID。',
        second_batch.results[0].transcript_id, item.transcript_id,
    )
    transcript_count = len(list_transcripts_for_video(db, 'ok'))
    report_equal(
        'Cross-source processing does not duplicate transcripts',
        '相同 video/language/source_kind 的第二次处理保持 transcript upsert 幂等。',
        transcript_count, 1,
    )

    terminal_source = 'https://www.youtube.com/@terminal/videos'
    discover(db, terminal_source, 'none', 'members', 'failed')
    def terminal_processor(url, **kwargs):
        video_id = url.split('=')[-1]
        if video_id == 'none': raise TubeScribeNoSubtitlesError('none')
        if video_id == 'members': raise TubeScribeMembersOnlyError('members')
        raise TubeScribeProcessingError('failed')
    terminal = process_pending_videos(db, raw, cleaned, limit=3, source_url=terminal_source, video_ids=['none','members','failed'], processor=terminal_processor)
    terminal_statuses = {result.video_id: result.status for result in terminal.results}
    report_equal(
        'Processor exceptions map to specific terminal outcomes',
        '无字幕、会员专享和普通处理错误分别记录为各自状态，不只检查状态集合。',
        terminal_statuses,
        {'none': 'no_subtitles', 'members': 'members_only', 'failed': 'failed'},
    )
    terminal_transcripts = {
        video_id: get_transcript(db, video_id)
        for video_id in ('none', 'members', 'failed')
    }
    report_equal(
        'Non-success outcomes do not create transcripts',
        '三个非成功分支都不能留下权威 transcript 记录。',
        terminal_transcripts, {'none': None, 'members': None, 'failed': None},
    )

    rollback_source = 'https://www.youtube.com/@rollback/videos'
    discover(db, rollback_source, 'rollback')
    connection = sqlite3.connect(db)
    connection.execute("CREATE TRIGGER reject_test_transcript BEFORE INSERT ON transcripts BEGIN SELECT RAISE(ABORT, 'test rejection'); END")
    connection.commit(); connection.close()
    try:
        process_pending_videos(db, raw, cleaned, source_url=rollback_source, video_ids=['rollback'], processor=success_processor)
    except StateStorageError as error:
        report_equal(
            'Transcript database rejection is surfaced',
            'SQLite trigger 拒绝 transcript 写入时，workflow 明确抛出 StateStorageError。',
            type(error).__name__, 'StateStorageError',
        )
    else:
        raise AssertionError('transcript write failure unexpectedly succeeded')
    connection = sqlite3.connect(db)
    try:
        rollback_job = connection.execute("SELECT status, transcript_id FROM processing_records WHERE video_id='rollback'").fetchone()
        report_equal(
            'Failed transcript write rolls back job success',
            'transcript 保存失败后，processing record 仍为 pending 且没有 transcript_id。',
            rollback_job, ('pending', None),
        )
    finally:
        connection.close()
    report_equal(
        'Failed transcript write leaves no partial transcript',
        '同一事务回滚后，数据库中不存在 rollback 视频的半成品 transcript。',
        get_transcript(db, 'rollback'), None,
    )
finish_section('Processing integration', 11)



=== Processing integration ===
范围: 用 fake TubeScribe 验证成功落库、跨来源幂等、业务终态和数据库写入失败回滚。

[PASS] Successful processing links a saved transcript
  验证: fake processor 成功后，job 标为 succeeded，并明确链接已保存的 transcript。
  实际: {'status': 'succeeded', 'transcript_saved': True, 'has_transcript_id': True}
  期望: {'status': 'succeeded', 'transcript_saved': True, 'has_transcript_id': True}

[PASS] Raw VTT path is stored relative to output root
  验证: 数据库/结果只保存可迁移的 raw 相对路径，不保存临时目录绝对路径。
  实际: 'raw/ok.zh.vtt'
  期望: 'raw/ok.zh.vtt'

[PASS] Cleaned TXT content becomes authoritative database text
  验证: fake processor 写出的 cleaned TXT 会被读取并保存到 SQLite transcript。
  实际: '清理后的正文'
  期望: '清理后的正文'

[PASS] Processed transcript stores the correct character count
  验证: processing workflow 保存的 character_count 与数据库正文字符数一致。
  实际: 6
  期望: 6

[PASS] The same video in another source reuses its transcript
  验证: 同一 video_id 从第二个来源处理时链接相同的全局 transcript ID。
  实际: 1
  期望: 1

[PASS] Cross-source processing does not duplicate transcripts
 


[PASS] Non-success outcomes do not create transcripts
  验证: 三个非成功分支都不能留下权威 transcript 记录。
  实际: {'none': None, 'members': None, 'failed': None}
  期望: {'none': None, 'members': None, 'failed': None}

[PASS] Transcript database rejection is surfaced
  验证: SQLite trigger 拒绝 transcript 写入时，workflow 明确抛出 StateStorageError。
  实际: 'StateStorageError'
  期望: 'StateStorageError'

[PASS] Failed transcript write rolls back job success
  验证: transcript 保存失败后，processing record 仍为 pending 且没有 transcript_id。
  实际: ('pending', None)
  期望: ('pending', None)

[PASS] Failed transcript write leaves no partial transcript
  验证: 同一事务回滚后，数据库中不存在 rollback 视频的半成品 transcript。
  实际: None
  期望: None

✅ Processing integration: 11/11 tests passed.
